### 1. 标准的softmax

In [53]:
import numpy as np
def softmax(x: np.ndarray):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

> 当 x 值过大，比如1000时， exp(x) 会上溢

### 2. 减去最大值来稳定数值

In [54]:
def minus_max(x: np.ndarray):
    vmax = np.max(x, axis=-1, keepdims=True)
    return x - vmax

def stable_softmax(x: np.ndarray):
    norm_x = minus_max(x)
    return softmax(norm_x)

### 3. 使用 log_softmax 稳定数值
> 不能直接 log(stable_softmax)
>
> 因为会存在 log(0) = -inf
>
> log_softmax(x) = x - log∑ex
>
> 再添加上 stable_softmax : x = x - vmax
>
> 此时，log∑ex = log∑e(x - vmax + vmax) = log∑ e(x - vmax) * e_vmax 
> 
> = log {e_vmax * ∑e(x - vmax)}  # 乘法结合律
> 
> = log e_vmax + log∑e(x - vmax)
>
> = vmax + log∑e(x - vmax)
>
> 那么：log_softmax(x) = x - vmax - log∑e(x - vmax)

In [55]:
def log_softmax(x: np.ndarray):
    norm_x = minus_max(x)
    log_sum_exps = np.log(np.sum(np.exp(norm_x), axis=-1, keepdims=True))
    return norm_x - log_sum_exps

In [56]:
x = np.array([[1000., 1001., 1002.], [1., 2., 3.]])
stable_sf = stable_softmax(x)
log_sf = log_softmax(x)
print(f"x shape: {x.shape}")
print(f"stable_softmax shape: {stable_sf.shape}")
print(stable_sf)
print(f"log_softmax shape: {log_sf.shape}")  
print(np.exp(log_sf))   # 应该和 stable_softmax 很接近

x shape: (2, 3)
stable_softmax shape: (2, 3)
[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]]
log_softmax shape: (2, 3)
[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]]


### 4. top-k
> 只保留每行 logits 最大的 k 个，其余设成 -inf，后面再 softmax。

In [57]:
# 先实现二分找到前k个最大值

# 找第k大的数
def quick_choose(a: np.ndarray, a_idxs: np.ndarray, l: int, r: int, k: int):
    """
    a: 当前搜索的数组，一维
    l: 当前搜索范围的左边界
    r: 当前搜索范围的右边界
    k: 当前要找的范围内前k个数
    """
    if l == r:
        return a[l]

    x = a[(l + r) >> 1]
    i = l - 1
    j = r + 1
    while i < j:
        i += 1
        while a[i] > x:
            i += 1
        j -= 1
        while a[j] < x:
            j -= 1
        if i < j:
            a[i], a[j] = a[j], a[i]
            a_idxs[i], a_idxs[j] = a_idxs[j], a_idxs[i]
    # 查完之后 i == j
    # 且 j 左边都大于x，右边都小于x
    dis = j - l + 1
    if dis >= k:
        return quick_choose(a, a_idxs, l, j, k)
    return quick_choose(a, a_idxs, j + 1, r, k - dis)

a = np.array([3, 1, 6, 9, 0, 2, 10])
a_idxs = np.arange(0, a.shape[0])
print(quick_choose(a, a_idxs, 0, len(a) - 1, 3))
print(a)
print(a_idxs)


6
[10  9  6  3  2  1  0]
[6 3 2 0 5 1 4]


```python

    i = l
    j = r
    ...
    while a[i] > x:
        i += 1
    while a[j] < x:
        j -= 1
```
这样写会导致 a[i] == a[j] 时陷入死循环，避免办法（写成 do-while 形式）：

```python

    i = l - 1
    j = r + 1
    ...
    while True:
        i += 1
        if a[i] > x:
            continue
        else:
            break
    while True:
        j -= 1
        if a[j] < x:
            continue
        else:
            break
```
或是更简洁的写法：
```python

    i = l - 1
    j = r + 1
    ...
    i += 1
    while a[i] > x:
        i += 1
    j -= 1
    while a[j] < x:
        j -= 1
```

In [58]:

def keep_top_k_logits(x: np.ndarray, k: int = 3):
    n_sampels = x.shape[0]
    res = np.full_like(x, fill_value=-float("inf"), dtype=np.float64)
    # 每个样本进行筛选k
    for i in range(n_sampels):
        a = x[i].copy()
        a_idxs = np.arange(0, a.shape[0])
        
        quick_choose(a, a_idxs, 0, a.shape[0] - 1, k)
        topk_idxs = a_idxs[:k]          # 只取前 k 个索引
        res[i][topk_idxs] = x[i][topk_idxs]
    
    return res

In [59]:
x = np.array([[1000., 1001., 1003., 1002., 1008.], [1., 5., 2., 0., 3.]])

print(x)
print(keep_top_k_logits(x, 3))

[[1.000e+03 1.001e+03 1.003e+03 1.002e+03 1.008e+03]
 [1.000e+00 5.000e+00 2.000e+00 0.000e+00 3.000e+00]]
[[ -inf  -inf 1003. 1002. 1008.]
 [ -inf    5.    2.  -inf    3.]]


In [60]:
# 或者使用 np.argpartition 获取前k个最大的logit值
def top_k(logits, k):
    logits = np.asarray(logits, dtype=np.float64)
    V = logits.shape[-1]
    k = min(k, V)

    idx = np.argpartition(logits, -k, axis=-1)[..., -k:]
    vals = np.take_along_axis(logits, idx, axis=-1)

    order = np.argsort(-vals, axis=-1)
    idx_sorted = np.take_along_axis(idx, order, axis=-1)
    vals_sorted = np.take_along_axis(vals, order, axis=-1)

    return vals_sorted, idx_sorted

### 5. top-p / nucleus

> 按概率从大到小排序，累计概率达到 p 就截断。
> 
> 要保留“第一个让累计概率超过 p 的 token”，并且至少保留一个。

先算概率 probs = stable_softmax(logits)

按概率降序排序

对排序后的概率做 cumsum

构造 remove mask

把移除的位置概率设 0

scatter 回原顺序

重新归一化

In [ ]:
def keep_top_p_logits(x: np.ndarray, p: float=0.8):
    """
    x: logits（未归一化）
    p: 目标累计概率阈值
    """
    probs = stable_softmax(x)
    # 先获取排序后的下标（从大到小）
    sorted_idx = np.argsort(-probs, axis=-1)
    # 利用排序后的下标，对原始 probs 进行排序
    sorted_probs = np.take_along_axis(probs, sorted_idx, axis=-1)
    # 从大到小累加 prob
    cum = np.cumsum(sorted_probs, axis=-1)
    # 根据 p 生成 remove-mask
    remove_mask = cum > p
    # 注意如果累加到i，刚好超过p，i一般而言需要被保留，所以需要做移位处理
    remove_mask[..., 1:] = remove_mask[..., :-1].copy()
    # 不管如何第一个概率肯定保留
    remove_mask[..., 0] = False
    
    keep_mask = ~remove_mask
    # 将 sort 之后的保留下来的 x 还原到原始的位置
    keep_x = np.zeros_like(x, dtype=bool)
    np.put_along_axis(keep_x, sorted_idx, keep_mask, axis=-1)
    return np.where(keep_x, x, -np.inf)

### 6. 组合：温度 + top-k + top-p + 采样

In [ ]:
def sample(x: np.ndarray, k: int=None, p: float=None, temperature=1.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    
    # 温度控制
    # temperature > 1 -> logits 分布更平滑，随机性越强
    # temperature < 1 -> logits 分布更尖凸，随机性越弱
    logits = x.astype(np.float64) / temperature
    
    # 先进行 top-k 过滤
    if k is not None:
        logits = keep_top_k_logits(logits, k)
        print("after top-k : ")
        print(*logits)
    
    # 再进行 top-p 过滤
    if p is not None:
        logits = keep_top_p_logits(logits, p)
        print("after top-p : ")
        print(*logits)
    
    # logit -> prob
    probs = stable_softmax(logits)
    print("probs:")
    print(*probs)
    
    # 采样
    flat_probs = probs.reshape(-1, probs.shape[-1])
    u = rng.random((flat_probs.shape[0], 1))
    
    cum = np.cumsum(flat_probs, axis=-1)
    idx = (cum >= u).argmax(axis=-1)

    return idx.reshape(probs.shape[:-1])

In [74]:
logits = np.array([[2.0, 1.0, 0.1, -1.0, 10, -100, 25, 30, 20, -10, -20, 15]])
idxs = sample(logits, k=8, p=None)

for i, idx in enumerate(idxs):
    print(f"sample {i}: idx={idx}, logit={logits[i, idx]}")

after top-k : 
[ 2.   1.   0.1 -inf 10.  -inf 25.  30.  20.  -inf -inf 15. ]
probs:
[6.86781125e-13 2.52652656e-13 1.02720905e-13 0.00000000e+00
 2.04726568e-09 0.00000000e+00 6.69254707e-03 9.93262053e-01
 4.50940274e-05 0.00000000e+00 0.00000000e+00 3.03841167e-07]
sample 0: idx=7, logit=30.0


In [77]:
x = np.array([[3., 2., 1., 0.]])
idxs = sample(x, k=None, p=0.7)

for i, idx in enumerate(idxs):
    print(f"sample {i}: idx={idx}, logit={logits[i, idx]}")

after top-p : 
[  3.   2. -inf -inf]
probs:
[0.73105858 0.26894142 0.         0.        ]
sample 0: idx=0, logit=2.0
